# Quick CPU and Memory Benchmark\n\nThis notebook runs two short benchmarks (CPU and RAM) and prints simple scores.\nTotal runtime target: under 1 minute.

In [ ]:
import os\nimport platform\nfrom time import perf_counter\n\nprint(f"Python: {platform.python_version()}")\nprint(f"Platform: {platform.platform()}")\nprint(f"CPU cores (logical): {os.cpu_count()}")

In [ ]:
def cpu_benchmark(duration_s=12.0, chunk=250_000):\n    """Single-thread integer benchmark using xorshift operations."""\n    start = perf_counter()\n    end = start + duration_s\n    ops = 0\n    x = 0x12345678\n\n    while perf_counter() < end:\n        for _ in range(chunk):\n            x ^= (x << 13) & 0xFFFFFFFF\n            x ^= x >> 17\n            x ^= (x << 5) & 0xFFFFFFFF\n        ops += chunk * 3\n\n    elapsed = perf_counter() - start\n    mops = ops / elapsed / 1_000_000\n    return {"elapsed": elapsed, "mops": mops, "state": x}\n\ncpu = cpu_benchmark()\nprint(f"CPU benchmark time: {cpu['elapsed']:.2f}s")\nprint(f"CPU throughput: {cpu['mops']:.2f} Mops/s")\nprint(f"CPU state checksum: {cpu['state']}")

In [ ]:
def memory_benchmark(size_mb=64, repeats=6):\n    """Sequential write/read benchmark over a byte buffer."""\n    size = size_mb * 1024 * 1024\n    buf = bytearray(size)\n    pattern = b"\xA5" * size\n\n    t0 = perf_counter()\n    for _ in range(repeats):\n        buf[:] = pattern\n    write_s = perf_counter() - t0\n\n    t1 = perf_counter()\n    checksum = 0\n    for _ in range(repeats):\n        checksum ^= buf.count(0xA5)\n    read_s = perf_counter() - t1\n\n    total_bytes = size * repeats\n    write_gbps = total_bytes / write_s / (1024 ** 3)\n    read_gbps = total_bytes / read_s / (1024 ** 3)\n\n    return {\n        "elapsed": write_s + read_s,\n        "write_gbps": write_gbps,\n        "read_gbps": read_gbps,\n        "checksum": checksum,\n    }\n\nmem = memory_benchmark()\nprint(f"Memory benchmark time: {mem['elapsed']:.2f}s")\nprint(f"Memory write throughput: {mem['write_gbps']:.2f} GiB/s")\nprint(f"Memory read throughput: {mem['read_gbps']:.2f} GiB/s")\nprint(f"Memory checksum: {mem['checksum']}")\n\nscore = cpu['mops'] + ((mem['write_gbps'] + mem['read_gbps']) / 2.0) * 100\nprint(f"\nSimple combined score (higher is better): {score:.2f}")